# Process plate counts to get ratios of variants in initial pool

An initial pooled library was made by adding equal volumes of all variants and then infecting this pool on MDCK-SIAT1 cells. These infections were done by serially diluting a starting 50 uL volume of pool. Barcodes were then isolated and sequenced so that we can determine representation of each variant in the pool, as well as the appropriate library pool dilution (i.e., MOI) to use in neutralization assays. 

The plots generated by this notebook are interactive, so you can mouseover points for details, use the mouse-scroll to zoom and pan, and use interactive dropdowns at the bottom of the plots.

## Setup
Import Python modules:

In [1]:
import pickle
import sys

import altair as alt

import matplotlib.pyplot as plt

import numpy

import pandas as pd
from os.path import join
import os
import ruamel.yaml as yaml

_ = alt.data_transformers.disable_max_rows()

## Add input data locations
Some of these files are defined as data, and some of these files are generated by running the specified library pooling data as `miscellaneous_plates` through the `seqneut-pipeline`. For details on how these files are generated, see the `README.md' in [https://github.com/jbloomlab/seqneut-pipeline](https://github.com/jbloomlab/seqneut-pipeline)

## Define inputs

In [2]:
# Define file path prefix
filepath_prefix = '/fh/fast/bloom_j/computational_notebooks/afarrell/2025/flu_seqneut_with_Ellebedy_lab'

# Viral library contents and barcode IDs
viral_library_csv = filepath_prefix + '/data/viral_libraries/H3_H1_ellebedy_library.csv'
# Neutralization standard set of barcode IDs
neut_standard_set_csv = filepath_prefix + '/data/neut_standard_sets/loes2023_neut_standards.csv'
# All samples included in library pooling sequencing run
# Contains information on library, dilution factor, R1 location
samplesfile = filepath_prefix + '/data/miscellaneous_plates/2025-05-08_H3H1_ellebedy_library_repool_samples.csv'

# Counts and fates files output by running library pooling samples as miscellaneous plates
platedir = filepath_prefix + '/results/miscellaneous_plates/20250509_repool_H3H1_ellebedy_library/'

library_mapping_file = filepath_prefix + '/data/miscellaneous_plates/library_mapping_file.csv'

# Identify all counts and fates CSVs
count_csvs = []
fate_csvs = []
file_list = os.listdir(platedir)
for f in file_list:
    location = platedir + f
    if "_counts" in f:
        count_csvs.append(location)
    elif "_fates" in f:
        fate_csvs.append(location)

#count_csvs

In [3]:
# Define a samples dataframe using the samples file
samples_df = pd.read_csv(samplesfile)
samples_df.drop(columns=['fastq'], inplace=True)
samples_df['sample'] = samples_df.apply(
    lambda x: '-'.join(x.astype(str)), axis=1
)

samples = samples_df["sample"].unique().tolist()
print(f"There are {len(samples)} barcode runs.")

There are 24 barcode runs.


## Statistics on barcode-parsing for each sample
Make interactive chart of the "fates" of the sequencing reads parsed for each sample on the plate.

If most sequencing reads are not "valid barcodes", this could potentially indicate some problem in the sequencing or barcode set you are parsing.

Potential fates are:
 - *valid barcode*: barcode that matches a known virus or neutralization standard, we hope most reads are this.
 - *invalid barcode*: a barcode with proper flanking sequences, but does not match a known virus or neutralization standard. If you  have a lot of reads of this type, it is probably a good idea to look at the invalid barcode CSVs (in the `./results/barcode_invalid/` subdirectory created by the pipeline) to see what these invalid barcodes are.
 - *unparseable barcode*: could not parse a barcode from this read as there was not a sequence of the correct length with the appropriate flanking sequence.
 - *low quality barcode*: low-quality or `N` nucleotides in barcode, could indicate problem with sequencing.
 - *failed chastity filter*: reads that failed the Illumina chastity filter, if these are reported in the FASTQ (they may not be).

Also, if the number of reads per sample is very uneven, that could indicate that you did not do a good job of balancing the different samples in the Illumina sequencing.

In [4]:
fates = (
    pd.concat([
        pd.read_csv(f).assign(well=os.path.basename(f).replace('_fates.csv', '')) 
        for f, s in zip(fate_csvs, samples)
    ])
    .merge(samples_df, validate="many_to_one", on="well")
    .assign(
        fate_counts=lambda x: x.groupby("fate")["count"].transform("sum"),
        sample_well=lambda x: x["sample"] + " (" + x["well"] + ")",
    )
    .query("fate_counts > 0")[  # only keep fates with at least one count
        ["fate", "count", "well", "sample_well", "dilution_factor"]
    ]
)

assert len(fates) == len(fates.drop_duplicates())


sample_wells = list(
    fates.sort_values(["dilution_factor"])["sample_well"]
)



fates_chart = (
    alt.Chart(fates)
    .encode(
        alt.X("count", scale=alt.Scale(nice=False, padding=3)),
        alt.Y(
            "sample_well",
            title=None,
            sort=sample_wells,
        ),
        alt.Color("fate", sort=sorted(fates["fate"].unique(), reverse=True)),
        alt.Order("fate", sort="descending"),
        tooltip=fates.columns.tolist(),
    )
    .mark_bar(height={"band": 0.85})
    .properties(
        height=alt.Step(10),
        width=200,
        title=f"Barcode parsing for initial titering plate",
    )
    .configure_axis(grid=False)
)

fates_chart

alt.Chart(...)

## Read barcode counts
Read the counts per barcode:

In [5]:
viral_df = pd.read_csv(viral_library_csv, delimiter=",")
viral_df
print(viral_df.columns)

Index(['barcode', 'strain'], dtype='object')


In [6]:
# get barcode counts
counts = (
    pd.concat([
        pd.read_csv(c).assign(well=os.path.basename(c).replace('_counts.csv', '')) 
        for c, s in zip(count_csvs, samples)
    ])
    .merge(samples_df, validate="many_to_one", on="well")
    .drop(columns=["replicate"])
    .assign(sample_well=lambda x: x["sample"] + " (" + x["well"] + ")")
)

# classify barcodes as viral or neut standard
barcode_class = pd.concat(
    [
        pd.read_csv(viral_library_csv)[["barcode", "strain"]].assign(
            neut_standard=False,
        ),
        pd.read_csv(neut_standard_set_csv)[["barcode"]].assign(
            neut_standard=True,
            strain=pd.NA,
        ),
    ],
    ignore_index=True,
)

# merge counts and classification of barcodes
assert set(counts["barcode"]) == set(barcode_class["barcode"])
counts = counts.merge(barcode_class, on="barcode", validate="many_to_one")

assert set(sample_wells) == set(counts["sample_well"])

## Average counts per barcode in each well

Plot average counts per barcode.
If a sample has inadequate barcode counts, it may not have good enough statistics for accurate analysis, and a QC-threshold is applied:

In [7]:
avg_barcode_counts = (
    counts.groupby(
        ["well", "sample_well"],
        dropna=False,
        as_index=False,
    )
    .aggregate(avg_count=pd.NamedAgg("count", "mean"))
    .assign(
        fails_qc=lambda x: (
            x["avg_count"] < 500
        ),
    )
)

avg_barcode_counts_chart = (
    alt.Chart(avg_barcode_counts)
    .encode(
        alt.X(
            "avg_count",
            title="average barcode counts per well",
            scale=alt.Scale(nice=False, padding=3),
        ),
        alt.Y("sample_well", sort=sample_wells),
        alt.Color(
            "fails_qc",
            title=f"fails {'min barcode count threshold'=}",
            legend=alt.Legend(titleLimit=500),
        ),
        tooltip=[
            alt.Tooltip(c, format=".3g") if avg_barcode_counts[c].dtype == float else c
            for c in avg_barcode_counts.columns
        ],
    )
    .mark_bar(height={"band": 0.85})
    .properties(
        height=alt.Step(10),
        width=250,
        title=f"Average barcode counts per well for titering plate",
    )
    .configure_axis(grid=False)
)

display(avg_barcode_counts_chart)

# drop wells failing QC
avg_barcode_counts_per_well_drops = list(avg_barcode_counts.query("fails_qc")["well"])

(avg_barcode_counts_chart).save(filepath_prefix + "/non-pipeline_analyses/library_pooling/figures/ave_barcodes_per_well.png")

alt.Chart(...)

## Fraction of counts from neutralization standard
Determine the fraction of counts from the neutralization standard in each sample, and make sure this fraction passess the QC threshold.

In [8]:
neut_standard_fracs = (
    counts.assign(
        neut_standard_count=lambda x: x["count"] * x["neut_standard"].astype(int)
    )
    .groupby(
        ["well", "sample_well", 'dilution_factor'],
        dropna=False,
        as_index=False,
    )
    .aggregate(
        total_count=pd.NamedAgg("count", "sum"),
        neut_standard_count=pd.NamedAgg("neut_standard_count", "sum"),
    )
    .assign(
        neut_standard_frac=lambda x: x["neut_standard_count"] / x["total_count"],
        fails_qc=lambda x: (
            x["neut_standard_frac"] < 0.001
        ),
    )
)

neut_standard_fracs_chart = (
    alt.Chart(neut_standard_fracs)
    .encode(
        alt.X(
            "neut_standard_frac",
            title="frac counts from neutralization standard per well",
            scale=alt.Scale(nice=False, padding=3),
        ),
        alt.Y("sample_well", sort=sample_wells),
        alt.Color(
            "fails_qc",
            title=f"fails {'min_neut_standard_frac_per_well'=}",
            legend=alt.Legend(titleLimit=500),
        ),
        tooltip=[
            alt.Tooltip(c, format=".3g") if neut_standard_fracs[c].dtype == float else c
            for c in neut_standard_fracs.columns
        ],
    )
    .mark_bar(height={"band": 0.85})
    .properties(
        height=alt.Step(10),
        width=250,
        title=f"Neutralization-standard fracs per well for titering plate, initial pool",
    )
    .configure_axis(grid=False)
    .configure_legend(titleLimit=1000)
)

display(neut_standard_fracs_chart)

# drop wells failing QC
min_neut_standard_frac_per_well_drops = list(
    neut_standard_fracs.query("fails_qc")["well"]
)

alt.Chart(...)

In [9]:
#Set up well groups for coloring in next plot
def well_group(well):
    first_letter = well[0]  # Get the first character
    if first_letter in ['A', 'B']:
        return '1.5e5 cells/well (A & B)'
    elif first_letter in ['C', 'D']:
        return '2e5 cells/well (C & D)'
    elif first_letter in ['E', 'F']:
        return '2.5e5 cells/well (E & F)'
    else:
        return 'Other'
        
neut_standard_fracs['well_group'] = neut_standard_fracs['well'].apply(well_group)

neut_standard_fracs

,well,sample_well,dilution_factor,total_count,neut_standard_count,neut_standard_frac,fails_qc,well_group
0,A1,A1-none-2-1 (A1),2,668689,34387,0.051425,False,1.5e5 cells/well (A & B)
1,A10,A10-none-1024-1 (A10),1024,919369,857763,0.932991,False,1.5e5 cells/well (A & B)
2,A11,A11-none-2048-1 (A11),2048,1148459,1117369,0.972929,False,1.5e5 cells/well (A & B)
3,A12,A12-none-4096-1 (A12),4096,1269442,1228118,0.967447,False,1.5e5 cells/well (A & B)
4,A2,A2-none-4-1 (A2),4,494837,25445,0.051421,False,1.5e5 cells/well (A & B)
5,A3,A3-none-8-1 (A3),8,1137936,105816,0.092989,False,1.5e5 cells/well (A & B)
6,A4,A4-none-16-1 (A4),16,804414,168460,0.209420,False,1.5e5 cells/well (A & B)
7,A5,A5-none-32-1 (A5),32,877126,304523,0.347183,False,1.5e5 cells/well (A & B)
8,A6,A6-none-64-1 (A6),64,902078,457243,0.506877,False,1.5e5 cells/well (A & B)
9,A7,A7-none-128-1 (A7),128,1040457,625289,0.600975,False,1.5e5 cells/well (A & B)


In [10]:
# Scatterplot of the same data as above, plotted by dilution factor
neut_standard_fracs_plot = alt.Chart(neut_standard_fracs).mark_circle(size=60).encode(
    alt.X('dilution_factor:Q', 
          scale=alt.Scale(type='log'),
          title='library pool reciprocal dilution factor'),
    alt.Y('neut_standard_frac:Q', 
          scale=alt.Scale(type='log'),
          title='fraction of reads = neutralization standard'),
    color='well_group:N', 
    tooltip=['well', 'dilution_factor', 'neut_standard_frac', 'total_count']
).interactive()

(neut_standard_fracs_plot).save(filepath_prefix + "/non-pipeline_analyses/library_pooling/figures/neut_standard_fracs_plot.png")
neut_standard_fracs_plot

alt.Chart(...)

## Rebalancing strains contained in the library
Viruses were rescued and blind passaged individually. To make the initial pool, we added equal volumes of all strains together and re-infected MDCK-SIAT1 cells. Now we can assess the contribution of each strain to the pool, and determine how much should be added of each virus to achieve more equal balancing. 

Each of the 3 viral barcodes associated with each strain were pooled prior to rescue, so they cannot be balanced. 

In [11]:
# Get summed barcode counts for all strains across all wells
straincounts_allbarcodes = (counts.groupby(['sample','sample_well','strain','dilution_factor','serum','well'])
                          .sum()
                          .reset_index()
                          .drop(columns = ['sample_well', 'neut_standard', 'barcode'])
                         )

# Get sum of all virus/barcode counts per well
sumperwell = (straincounts_allbarcodes.groupby(['sample','dilution_factor','serum','well'])
              .sum()
              .drop(columns=['strain'])
              .reset_index()
              .rename(columns={'count':'counts_perwell'})
             )

# Merge dataframes and calculate fraction of each well devoted to each strain
merged_df = straincounts_allbarcodes.merge(sumperwell, on=['sample','dilution_factor','serum','well'])
merged_df['fraction_strain'] = merged_df['count'] /merged_df['counts_perwell'] / 2
merged_df

,sample,strain,dilution_factor,serum,well,count,counts_perwell,fraction_strain
0,A1-none-2-1,A/AbuDhabi/6753/2023_H3N2,2,none,A1,3982,634302,0.003139
1,A1-none-2-1,A/Arizona/IVYG43Q65T3/2023_H1N1,2,none,A1,10718,634302,0.008449
2,A1-none-2-1,A/Bangkok/P3599/2023_H3N2,2,none,A1,5648,634302,0.004452
3,A1-none-2-1,A/Bangkok/P3755/2023_H3N2,2,none,A1,6541,634302,0.005156
4,A1-none-2-1,A/Bhutan/0006/2023_H3N2,2,none,A1,6673,634302,0.005260
...,...,...,...,...,...,...,...,...
2323,B9-none-512-2,A/Victoria/4897/2022_R142K_H1N1,512,none,B9,5477,111874,0.024478
2324,B9-none-512-2,A/Wisconsin/27/2023_H3N2,512,none,B9,0,111874,0.000000
2325,B9-none-512-2,A/Wisconsin/588/2019_H1N1,512,none,B9,4154,111874,0.018566
2326,B9-none-512-2,A/Wisconsin/67/2022_H1N1,512,none,B9,3000,111874,0.013408


We now have this fraction of reads devoted to all strains calculated for all wells. However, ideally we should just focus on those wells containing dilutions that we would use for actual neutralization assays. We should choose a set of replicate wells where the fraction of neutralization standard reads begins to increase linearly with the increasing reciprocal dilution factor. See plots above for choosing these wells. 

In [12]:
library_map=pd.read_csv(library_mapping_file).drop_duplicates().reset_index(drop=True)
library_map

,strain,library
0,A/Shaanxi-Yuyang/1204/2024_H3N2,19_strain
1,A/Oman/CPHL_7248310/2024_H3N2,19_strain
2,A/DistrictOfColumbia/27/2023_H3N2,19_strain
3,A/Lisboa/124/2024_H3N2,19_strain
4,A/BritishColumbia/RV00920/2023_H3N2,19_strain
...,...,...
92,A/Pennsylvania/51/2023_H1N1,19_strain
93,A/Victoria/1389/2023_H1N1,19_strain
94,A/Arizona/IVYG43Q65T3/2023_H1N1,19_strain
95,A/Jeonbuk/1544/2023_H1N1,19_strain


In [13]:
# Using A4 and B4, corresponding to reciprocal dilution factor = 256
single_well = merged_df.loc[merged_df['sample'].str.contains('A4-|B4-')]
single_well

,sample,strain,dilution_factor,serum,well,count,counts_perwell,fraction_strain
582,A4-none-16-1,A/AbuDhabi/6753/2023_H3N2,16,none,A4,3659,635954,0.002877
583,A4-none-16-1,A/Arizona/IVYG43Q65T3/2023_H1N1,16,none,A4,22001,635954,0.017298
584,A4-none-16-1,A/Bangkok/P3599/2023_H3N2,16,none,A4,3382,635954,0.002659
585,A4-none-16-1,A/Bangkok/P3755/2023_H3N2,16,none,A4,5918,635954,0.004653
586,A4-none-16-1,A/Bhutan/0006/2023_H3N2,16,none,A4,4781,635954,0.003759
...,...,...,...,...,...,...,...,...
1838,B4-none-16-2,A/Victoria/4897/2022_R142K_H1N1,16,none,B4,67017,1600718,0.020933
1839,B4-none-16-2,A/Wisconsin/27/2023_H3N2,16,none,B4,8361,1600718,0.002612
1840,B4-none-16-2,A/Wisconsin/588/2019_H1N1,16,none,B4,38973,1600718,0.012174
1841,B4-none-16-2,A/Wisconsin/67/2022_H1N1,16,none,B4,37282,1600718,0.011645


In [14]:
# Now add in a column from library_mapping so we can color the chart by 78- or 19-strain library
single_well_merged = pd.merge(single_well, library_map, on='strain')
single_well_merged

,sample,strain,dilution_factor,serum,well,count,counts_perwell,fraction_strain,library
0,A4-none-16-1,A/AbuDhabi/6753/2023_H3N2,16,none,A4,3659,635954,0.002877,78_strain
1,A4-none-16-1,A/Arizona/IVYG43Q65T3/2023_H1N1,16,none,A4,22001,635954,0.017298,19_strain
2,A4-none-16-1,A/Bangkok/P3599/2023_H3N2,16,none,A4,3382,635954,0.002659,78_strain
3,A4-none-16-1,A/Bangkok/P3755/2023_H3N2,16,none,A4,5918,635954,0.004653,78_strain
4,A4-none-16-1,A/Bhutan/0006/2023_H3N2,16,none,A4,4781,635954,0.003759,78_strain
...,...,...,...,...,...,...,...,...,...
189,B4-none-16-2,A/Victoria/4897/2022_R142K_H1N1,16,none,B4,67017,1600718,0.020933,19_strain
190,B4-none-16-2,A/Wisconsin/27/2023_H3N2,16,none,B4,8361,1600718,0.002612,78_strain
191,B4-none-16-2,A/Wisconsin/588/2019_H1N1,16,none,B4,38973,1600718,0.012174,19_strain
192,B4-none-16-2,A/Wisconsin/67/2022_H1N1,16,none,B4,37282,1600718,0.011645,19_strain


In [15]:
# Calculate mean fraction strain across both wells
mean_df = single_well_merged.groupby(['strain'])['fraction_strain'].mean().to_frame().rename(columns = {'fraction_strain': 'mean_fraction_strains'}).reset_index()
mean_single_well = single_well_merged.merge(mean_df, on = 'strain', how = 'left')

# calcualte ratios to add for equal pool
num_strains = 76
mean_single_well['ratio_to_add'] = (1/num_strains)/mean_single_well['fraction_strain']
mean_single_well['mean_ratio_to_add'] = (1/num_strains)/mean_single_well['mean_fraction_strains']

mean_single_well['est_tcid50'] = (mean_single_well['mean_fraction_strains']*25000)*76

mean_single_well.head(5)

,sample,strain,dilution_factor,serum,well,count,counts_perwell,fraction_strain,library,mean_fraction_strains,ratio_to_add,mean_ratio_to_add,est_tcid50
0,A4-none-16-1,A/AbuDhabi/6753/2023_H3N2,16,none,A4,3659,635954,0.002877,78_strain,0.003007,4.573827,4.375893,5713.119646
1,A4-none-16-1,A/Arizona/IVYG43Q65T3/2023_H1N1,16,none,A4,22001,635954,0.017298,19_strain,0.015613,0.760676,0.842776,29663.878030
2,A4-none-16-1,A/Bangkok/P3599/2023_H3N2,16,none,A4,3382,635954,0.002659,78_strain,0.003055,4.948442,4.307700,5803.561072
3,A4-none-16-1,A/Bangkok/P3755/2023_H3N2,16,none,A4,5918,635954,0.004653,78_strain,0.004762,2.827920,2.762891,9048.492071
4,A4-none-16-1,A/Bhutan/0006/2023_H3N2,16,none,A4,4781,635954,0.003759,78_strain,0.004490,3.500446,2.930688,8530.420010


## Visualize barcode- and strain-level balancing in the current pool

In [16]:

# Plot the current fraction of each strain in the pool
strains_chart = (
    alt.Chart(mean_single_well)
    .encode(
        alt.X(
            "fraction_strain",
            scale=alt.Scale(nice=False, padding=3, domain = (0, 0.05)),
        ),
        alt.Y("strain"),
        alt.Color("library", legend=alt.Legend(title="Library")),
        tooltip = ['strain', 'fraction_strain', 'est_tcid50'],
    )
).mark_bar(height={"band": 0.85}).properties(
        height=alt.Step(10),
        width=250,
        title="",
    ).properties(
        height = alt.Step(10),
        width = 200,
        title = "Strain representation, initial pool")

# add veritcal line where we would expect equal representation of all barcodes in pool
expected_line = alt.Chart(
    pd.DataFrame({'x': [1/97]})
).mark_rule(strokeDash = [2,2], strokeWidth = 2).encode(x = 'x')

(strains_chart + expected_line).save(filepath_prefix + "/non-pipeline_analyses/library_pooling/figures/repool_strains_chart.png")
#make a .csv with these fractions
mean_single_well.to_csv(filepath_prefix + "/non-pipeline_analyses/library_pooling/results/repool_strain_representation_A2_B2.csv", index=False)

# plot both barcode counts and expected line
strains_chart + expected_line

alt.LayerChart(...)

In [17]:
# Each barcode fraction across strains
all_barcode_counts = counts[['strain', 'barcode', 'count', 'well']].dropna()
single_well_all_barcode_counts = all_barcode_counts[all_barcode_counts['well'].isin(['A4','B4'])]

# Get tidy single well means
tidy_single_well = single_well_all_barcode_counts[['strain','barcode','count']].groupby(['strain', 'barcode']).mean().reset_index()
# Get sums for each strain
strain_sums_df = tidy_single_well.groupby('strain').sum().rename(columns = {'count': 'strain_count_sum'}).reset_index()
# Merge and calculate per strain the fraction represented by each barcode
tidy_single_well = tidy_single_well.merge(strain_sums_df[['strain', 'strain_count_sum']], 
                       on = ['strain'],
                       validate="many_to_one",
                      )
tidy_single_well['per_strain_fraction_barcode'] = tidy_single_well['count'] / tidy_single_well['strain_count_sum']
tidy_single_well['barcode_letter'] = (['A', 'B', 'C'] * len(strain_sums_df))

# Plot as colored bar chart
bar_chart = alt.Chart(tidy_single_well).mark_bar(height={"band": 0.85}).encode(
    x = 'per_strain_fraction_barcode',
    y = 'strain',
    color=alt.Color('barcode_letter', legend=None),
    tooltip = ['strain', 'per_strain_fraction_barcode', 'barcode'],
).configure_axis(grid=False).properties(
        height = alt.Step(10),
        width = 200,
        title = "Barcode fraction for each strain, initial pool")

bar_chart.save("../figures/repool_barcode_fraction_per_strain.png", dpi=300)
bar_chart

alt.Chart(...)

In [18]:
def get_barcode_counts(barcode, platedir):
    barcode_counts = {}

    for filename in os.listdir(platedir):
        if filename.endswith("_counts.csv"):
            filepath = os.path.join(platedir, filename)

            # Load CSV using pandas
            df = pd.read_csv(filepath)

            # Find the row where the barcode matches
            match = df[df["barcode"] == barcode]

            if not match.empty:
                count = match.iloc[0]["count"]
                barcode_counts[filename] = count

    return barcode_counts

In [19]:
barcode_5312 = "CAAAAATCTAATACTT"
results = get_barcode_counts(barcode_5312, platedir)

for fname, cnt in results.items():
    print(f"{fname}: {cnt}")

A8_counts.csv: 1288
B7_counts.csv: 3894
A11_counts.csv: 139
B3_counts.csv: 10240
B4_counts.csv: 15338
A5_counts.csv: 7656
B8_counts.csv: 2115
A12_counts.csv: 32
A3_counts.csv: 10976
B2_counts.csv: 21153
B1_counts.csv: 7984
B11_counts.csv: 115
A9_counts.csv: 439
A4_counts.csv: 7600
A1_counts.csv: 9322
A7_counts.csv: 2940
B6_counts.csv: 7895
B5_counts.csv: 6467
B9_counts.csv: 705
A6_counts.csv: 4115
B10_counts.csv: 49
A10_counts.csv: 288
A2_counts.csv: 5308
B12_counts.csv: 33


In [20]:
bad_barcode = 'CATAATCTATTGAAGC'
results = get_barcode_counts(bad_barcode, platedir)

for fname, cnt in results.items():
    print(f"{fname}: {cnt}")

A8_counts.csv: 0
B7_counts.csv: 0
A11_counts.csv: 0
B3_counts.csv: 0
B4_counts.csv: 0
A5_counts.csv: 0
B8_counts.csv: 0
A12_counts.csv: 0
A3_counts.csv: 0
B2_counts.csv: 0
B1_counts.csv: 0
B11_counts.csv: 0
A9_counts.csv: 0
A4_counts.csv: 0
A1_counts.csv: 4
A7_counts.csv: 0
B6_counts.csv: 0
B5_counts.csv: 0
B9_counts.csv: 0
A6_counts.csv: 0
B10_counts.csv: 0
A10_counts.csv: 0
A2_counts.csv: 0
B12_counts.csv: 0


In [21]:
barcode_5310 = 'TATGATGTCAGATAAA'
results = get_barcode_counts(barcode_5310, platedir)

for fname, cnt in results.items():
    print(f"{fname}: {cnt}")

A8_counts.csv: 0
B7_counts.csv: 0
A11_counts.csv: 0
B3_counts.csv: 33
B4_counts.csv: 12
A5_counts.csv: 36
B8_counts.csv: 0
A12_counts.csv: 0
A3_counts.csv: 26
B2_counts.csv: 34
B1_counts.csv: 35
B11_counts.csv: 0
A9_counts.csv: 0
A4_counts.csv: 36
A1_counts.csv: 22
A7_counts.csv: 32
B6_counts.csv: 18
B5_counts.csv: 0
B9_counts.csv: 0
A6_counts.csv: 28
B10_counts.csv: 0
A10_counts.csv: 0
A2_counts.csv: 42
B12_counts.csv: 0
